# Day 4 — Supervised Fine-Tuning (SFT)

Fine-tune a GPT-2 model on a cybersecurity instruction dataset using Hugging Face TRL's SFTTrainer. Then compare responses before and after training.

In [ ]:
import json
import os
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline
from trl import SFTTrainer

---
## Part 1: Load Dataset

Load the cybersecurity instruction dataset from Day 3, or create an inline version if unavailable.

In [ ]:
dataset_path = "../day-03-dataset-preparation/cybersecurity_dataset.jsonl"

if os.path.exists(dataset_path):
    with open(dataset_path) as f:
        raw_data = [json.loads(line) for line in f]
    print(f"Loaded {len(raw_data)} examples from Day 3 dataset")
else:
    raw_data = [
        {"instruction": "What is a firewall?", "response": "A firewall monitors and controls network traffic based on security rules."},
        {"instruction": "Explain encryption.", "response": "Encryption converts plaintext into ciphertext using a key."},
        {"instruction": "What is a DDoS attack?", "response": "A DDoS attack overwhelms a target with traffic from multiple sources."},
        {"instruction": "Define social engineering.", "response": "Social engineering manipulates people to give up information."},
        {"instruction": "What is symmetric vs asymmetric encryption?", "response": "Symmetric uses one key. Asymmetric uses a key pair."},
        {"instruction": "Explain zero-day vulnerability.", "response": "A zero-day is a flaw unknown to the vendor with no patch available."},
        {"instruction": "What is MFA?", "response": "MFA requires two or more verification factors for access."},
        {"instruction": "How does a VPN work?", "response": "A VPN encrypts traffic and hides your IP address from eavesdroppers."},
        {"instruction": "What is ransomware?", "response": "Ransomware encrypts files and demands payment for decryption."},
        {"instruction": "Define network segmentation.", "response": "Network segmentation divides a network into smaller parts for security."},
    ]
    print(f"Created inline dataset with {len(raw_data)} examples")

# Format for SFT
def format_for_sft(example):
    return {"text": f"Instruction:\n{example['instruction']}\n\nResponse:\n{example['response']}"}

dataset = Dataset.from_list([format_for_sft(ex) for ex in raw_data])
split = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Training:   {len(split['train'])} examples")
print(f"Validation: {len(split['test'])} examples")

---
## Part 2: Load Base Model (GPT-2)

In [ ]:
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Part 3: Test Before Fine-Tuning

See how GPT-2 responds to a cybersecurity question before any training.

In [ ]:
test_prompt = "Instruction:\nWhat is a firewall?\n\nResponse:\n"

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
result = pipe(test_prompt, max_new_tokens=60, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
response_before = result[0]["generated_text"][len(test_prompt):].strip()

print("Before SFT:")
print(f"  {response_before}")

---
## Part 4: Supervised Fine-Tuning

Run SFTTrainer on the cybersecurity dataset. This will update GPT-2's weights to produce better domain-specific responses.

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-cybersecurity-sft",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    max_steps=30,
    learning_rate=3e-5,
    logging_steps=5,
    eval_steps=10,
    save_steps=30,
    eval_strategy="steps",
    report_to="none",
    fp16=False,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    dataset_text_field="text",
    max_seq_length=256,
    response_template="Response:\n",
)

trainer.train()

---
## Part 5: Save and Reload

In [ ]:
save_path = "./gpt2-cybersecurity-sft"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved to {save_path}")

# Reload
ft_model = AutoModelForCausalLM.from_pretrained(save_path)
ft_tokenizer = AutoTokenizer.from_pretrained(save_path)

---
## Part 6: Test After Fine-Tuning

In [ ]:
pipe_after = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)
result = pipe_after(test_prompt, max_new_tokens=60, temperature=0.7, do_sample=True, pad_token_id=ft_tokenizer.eos_token_id)
response_after = result[0]["generated_text"][len(test_prompt):].strip()

print("After SFT:")
print(f"  {response_after}")

---
## Comparison

In [ ]:
print("Before SFT:")
print(f"  {response_before}")
print()
print("After SFT:")
print(f"  {response_after}")

## Key Takeaways

- SFT updates a model's weights to perform better on specific tasks
- TRL's SFTTrainer handles data formatting, loss masking, and training
- The `response_template` tells the trainer which tokens to learn from
- Fine-tuning on just 10 examples for 30 steps can produce noticeable improvement
- The fine-tuned model can be saved and reloaded for inference
- SFT is the foundation for more advanced techniques like LoRA and RLHF